# Calificar — Proyecto Oráculo

Corre una config sobre las 294 instancias del test y da una nota.
No es el notebook de los estudiantes: usa `datos_test.json` y los verificadores de IFBench.


## 1 · Instalar y clonar  ·  *al terminar, reinicia el entorno de ejecución*

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict syllapy
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct
!git clone -q https://github.com/allenai/IFBench

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-AyD-algoritmos/main"
# --no-cache: Colab a veces reusa un oraculo.py viejo.
!wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O calificar.py {REPO}/profesor/calificar.py
!wget -q --no-cache -O datos_test.json {REPO}/profesor/datos_test.json

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
# "stopwords" es lo único que no pide el notebook de estudiantes: lo usa
# ratio:stop_words, una familia del test que no está en datos_visibles.json.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
    ("corpora", "stopwords"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")


## 2 · Drive y modelo

`CARPETA` es el caché y los `entrega.json` de los grupos: `MyDrive/oraculo_profesor/`.
`datos_test.json` y `calificar.py` se bajan del repo en la celda 1.

**Usa el mismo alias con el que buscaron los grupos.** La clave de caché incluye el nombre del
modelo — una nota sacada con otro modelo no es comparable.

In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

CARPETA = Path("/content/drive/MyDrive/oraculo_profesor")
CARPETA.mkdir(parents=True, exist_ok=True)

from ayudas import cargar_modelo

#  "pequeno"     → Qwen/Qwen3-1.7B
#  "ministral3b" → mistralai/Ministral-3-3B-Instruct-2512-BF16
#  "llama3b"     → unsloth/Llama-3.2-3B-Instruct
#  "qwen8b"      → unsloth/Qwen3-8B-unsloth-bnb-4bit
#  "mistral7b"   → unsloth/mistral-7b-instruct-v0.3-bnb-4bit
modelo = cargar_modelo("pequeno")


## 3 · El oráculo del test

In [ ]:
from calificar import oraculo_test

oraculo, test = oraculo_test(
    modelo,
    "datos_test.json",
    cache=CARPETA / "cache_test.json",
)


## 4 · Calificar una config

Antes de lanzar las 294, mide con `n=8` y cronometra — de ahí sale el costo real por instancia
para extrapolar ×37. El caché queda en Drive: una desconexión no cuesta la corrida.

In [ ]:
from calificar import calificar

config = {"rol": 1, "estrategia": 2, "formato": 1, "verificacion": 1, "temperatura": 0.0}

r = calificar(oraculo, config, n=8)  # primero una muestra chica, cronometrada
# r = calificar(oraculo, config)     # las 294, cuando ya se sabe el costo


## 5 · Calificar a todos los grupos

Lee cada `entrega.json` bajo `CARPETA/"entregas"` y devuelve la tabla ordenada por precisión.
Dos grupos con la misma config sólo cuestan una pasada: la clave de caché no depende del grupo.

In [ ]:
from calificar import calificar_entregas

tabla = calificar_entregas(oraculo, CARPETA / "entregas")
